In [ ]:
import pandas as pd
import sqlite3 as db

In [ ]:
#Poplation data set clean up
pop = pd.read_csv("county_estimates.csv", sep=";")
pop.columns = (
    pop.columns
    .str.replace("\ufeff", "", regex=False)  # remove BOM if present
    .str.strip()
)

pop["FIPS"] = pop["FIPS"].astype(str).str.zfill(5)
pop["Year"] = pd.to_datetime(pop["Year"], errors="coerce").dt.year

pop = pop[
    (pop["Area Type"] == "County") &
    (pop["Estimate"] == "Annual Estimate")
][["FIPS", "Area", "Year", "value"]].rename(columns={"value": "population"})

##cleaning crash data set to filter for traffic accidents prepping to merge
try:
    tr = pd.read_csv("nc_transportation_line_all_data.csv")
except UnicodeDecodeError:
    # Some exports of this file are UTF-16 encoded; retry with explicit encoding.
    tr = pd.read_csv("nc_transportation_line_all_data.csv", encoding="utf-16")
tr["Geo_ID"] = tr["Geo_ID"].astype(str).str.zfill(5)

crashes = tr[
    (tr["Variable"] == "Traffic Accidents")
][["Geo_ID", "Area_Name", "Year", "Value"]].rename(
    columns={"Geo_ID": "FIPS", "Value": "crashes"}
)

#Actual merge of the two datasets
merged = crashes.merge(pop, on=["FIPS", "Year"], how="inner")

# Charlotte proxy due to one data set mixing charlotte and Mecklenburg County
df = merged[merged["FIPS"] == "37119"].copy()
df = df.sort_values("Year").drop_duplicates(subset=["Year"] )

#Growth variables for pop
df["pop_change"] = df["population"].diff()
df["pop_growth_pct"] = df["population"].pct_change() * 100
df["crash_growth_pct"] = df["crashes"].pct_change() * 100

model_df = df.dropna(subset=["pop_change", "crashes"]).copy()

# time-aware split
split_idx = int(len(model_df) * 0.8)
train = model_df.iloc[:split_idx]
test = model_df.iloc[split_idx:]